# AI-Fiber-NLC: Quickstart Demo

> **AI Nonlinear Compensation for Optical Fiber Systems**
>
> This notebook demonstrates the core concept: using AI to compensate for
> fiber nonlinearities that limit modern coherent optical transmission.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AI-Fiber-NLC/AI-Fiber-NLC/blob/main/notebooks/quickstart_demo.ipynb)

## What you will see

1. **Constellation diagrams** — visualize nonlinear distortion before and after compensation
2. **DBP baseline** — the classical digital back-propagation algorithm
3. **MLP-NLC** — a neural network approach for nonlinear compensation
4. **Performance comparison** — Q-factor improvement across all methods

## Setup

Install dependencies (only needed on first run):

In [ ]:
# Check if running in Colab and install dependencies
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install -q OptiCommPy torch numpy matplotlib
    !git clone -q https://github.com/AI-Fiber-NLC/AI-Fiber-NLC.git /content/AI-Fiber-NLC
    sys.path.insert(0, '/content/AI-Fiber-NLC')
    DATA_DIR = '/content/AI-Fiber-NLC/data/raw'
    MODEL_DIR = '/content/AI-Fiber-NLC/models'
else:
    DATA_DIR = './data/raw'
    MODEL_DIR = './models'

import numpy as np
import matplotlib.pyplot as plt
import torch

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['figure.dpi'] = 100
print('Setup complete!')

## Step 1: Load Simulation Data

We use pre-generated data from a 16QAM, 800km single-polarization fiber link
(MVB-1 scene). The signal has been distorted by chromatic dispersion and
Kerr nonlinearity, with EDFA amplification at each 80km span.

In [ ]:
from src.data.dataset import NLCDataset
from src.benchmark.protocol import SCENES, MVB1

# Load data at +1 dBm launch power (power index 4)
scene = MVB1
power_idx = 4
powers = np.linspace(scene.tx_power_range_dbm[0], scene.tx_power_range_dbm[1], scene.tx_power_points)
power_dbm = powers[power_idx]

ds = NLCDataset(DATA_DIR, scene.name, power_dbm)
print(f'Scene: {scene.name}')
print(f'  Modulation: {scene.modulation}')
print(f'  Distance: {scene.fiber_length_km * scene.num_spans:.0f} km')
print(f'  Power: {power_dbm:+.1f} dBm')
print(f'  Samples: {len(ds):,}')

# Load a subset for visualization
n_viz = 4000  # show 4000 symbols for clear constellation
rx_list, tx_list = [], []
for i in range(n_viz):
    rx_i, tx_i = ds[i]
    rx_list.append(rx_i)
    tx_list.append(tx_i)

rx_subset = torch.stack(rx_list)  # (N, 2)
tx_subset = torch.stack(tx_list)  # (N, 2)
print(f'  Loaded {n_viz:,} samples for visualization')

## Step 2: Visualize Constellation Diagrams

The constellation diagram shows how the transmitted 16QAM symbols are
distorted after 800km of fiber propagation.

In [ ]:
def plot_constellation(rx_iq, tx_iq, title='Constellation Diagram'):
    """Plot constellation with I/Q components."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # TX constellation
    tx_i = tx_iq[:, 0].numpy()
    tx_q = tx_iq[:, 1].numpy()
    axes[0].scatter(tx_i, tx_q, s=0.5, alpha=0.6, color='blue')
    axes[0].set_title('Transmitted (Ideal 16QAM)')
    axes[0].set_xlabel('I')
    axes[0].set_ylabel('Q')
    axes[0].axis('equal')
    axes[0].grid(True, alpha=0.3)

    # RX constellation
    rx_i = rx_iq[:, 0].numpy()
    rx_q = rx_iq[:, 1].numpy()
    axes[1].scatter(rx_i, rx_q, s=0.5, alpha=0.6, color='red')
    axes[1].set_title('Received (After 800km Fiber)')
    axes[1].set_xlabel('I')
    axes[1].set_ylabel('Q')
    axes[1].axis('equal')
    axes[1].grid(True, alpha=0.3)

    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Compute EVM-based Q estimate
    error = rx_iq - tx_iq
    sig_pow = torch.mean(torch.sum(tx_iq**2, dim=1)).item()
    err_pow = torch.mean(torch.sum(error**2, dim=1)).item()
    if err_pow > 0 and sig_pow > 0:
        evm = np.sqrt(err_pow / sig_pow)
        q_est = 20 * np.log10(1.0 / evm) if evm > 0 else float('inf')
        print(f'  Relative Q-factor: {q_est:+.2f} dB  (EVM: {evm:.4f})')

plot_constellation(rx_subset, tx_subset, 'Fiber Nonlinearity: TX vs RX')

## Step 3: DBP Baseline

Digital Back Propagation (DBP) is the classical approach to nonlinear
compensation. It reverses the fiber propagation by numerically solving
the nonlinear Schrodinger equation with inverted coefficients.

In [ ]:
from src.models.baseline_dbp import DBPCompensator
import time

print('Running DBP compensation...')
dbp = DBPCompensator(MVB1, steps_per_span=10)

# Load full dataset for DBP
ds_full = NLCDataset(DATA_DIR, scene.name, power_dbm)
rx_full_list, tx_full_list = [], []
for i in range(len(ds_full)):
    rx_full_list.append(ds_full[i][0])
    tx_full_list.append(ds_full[i][1])
rx_full = torch.stack(rx_full_list)
tx_full = torch.stack(tx_full_list)

t0 = time.time()
dbp_output = dbp.compensate(rx_full, launch_power_dbm=power_dbm)
dbp_time = time.time() - t0
print(f'DBP completed in {dbp_time:.1f}s')

# Visualize DBP result
plot_constellation(dbp_output[:n_viz], tx_full[:n_viz], 'DBP Compensation Result')

## Step 4: MLP-NLC (Neural Network)

Now we apply a pre-trained MLP model with temporal context window to
compensate for the fiber nonlinearities.

In [ ]:
from src.models.mlp_nlc import MLPWithMemory_NLC
import os

# Check if a trained model exists
model_path = os.path.join(MODEL_DIR, 'memory_MVB-1_p4.pt')

if os.path.exists(model_path):
    print('Loading pre-trained MLP model...')
    checkpoint = torch.load(model_path, map_location='cpu', weights_only=False)
    model = MLPWithMemory_NLC(
        memory_size=checkpoint['memory_size'],
        hidden_dims=checkpoint['hidden_dims'],
        dropout=checkpoint['dropout'],
    )
    model.load_state_dict(checkpoint['model_state'])
    model.eval()
    print(f'  Model: MLP with memory={checkpoint["memory_size"]}')
    print(f'  Parameters: {checkpoint["total_params"]:,}')
    print(f'  Best Q: {checkpoint["best_q"]:+.2f} dB')

    # Create windowed input
    mem = checkpoint['memory_size']
    window = 2 * mem + 1
    rx_windows = rx_full.unfold(0, window, 1)  # (N-2m, window, 2)
    tx_center = tx_full[mem: -mem] if mem > 0 else tx_full

    # Run inference
    print('Running MLP inference...')
    t0 = time.time()
    with torch.no_grad():
        mlp_output = model(rx_windows)
    mlp_time = time.time() - t0
    print(f'MLP inference completed in {mlp_time:.2f}s')

    # Visualize
    plot_constellation(mlp_output[:n_viz], tx_center[:n_viz], 'MLP-NLC Compensation Result')
else:
    print('No pre-trained model found.')
    print('To train: python scripts/train_mlp.py --scenario mvb1 --power-index 4')

## Step 5: Performance Comparison

Compare Q-factor improvement across all methods.

In [ ]:
def compute_q(compensated, tx):
    error = compensated - tx
    sig_pow = torch.mean(torch.sum(tx**2, dim=1)).item()
    err_pow = torch.mean(torch.sum(error**2, dim=1)).item()
    if err_pow > 0 and sig_pow > 0:
        evm = np.sqrt(err_pow / sig_pow)
        return 20 * np.log10(1.0 / evm) if evm > 0 else 99.9
    return 99.9

print('=' * 60)
print(f'Performance Comparison — {scene.name}, {power_dbm:+.1f} dBm')
print('=' * 60)

# Baseline
q_base = compute_q(rx_full, tx_full)
print(f'  Baseline (no comp):  {q_base:+.2f} dB')

# DBP
q_dbp = compute_q(dbp_output, tx_full)
print(f'  DBP (10 steps/span): {q_dbp:+.2f} dB  (improvement: {q_dbp - q_base:+.2f} dB)')

# MLP
if os.path.exists(model_path):
    q_mlp = compute_q(mlp_output, tx_center)
    print(f'  MLP (memory={mem}):    {q_mlp:+.2f} dB  (improvement: {q_mlp - q_base:+.2f} dB)')

print('=' * 60)

# Bar chart
methods = ['Baseline', 'DBP']
qs = [q_base, q_dbp]
colors = ['#e74c3c', '#3498db']

if os.path.exists(model_path):
    methods.append('MLP-NLC')
    qs.append(q_mlp)
    colors.append('#2ecc71')

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(methods, qs, color=colors, width=0.5, edgecolor='white', linewidth=2)
for bar, q in zip(bars, qs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{q:+.2f} dB', ha='center', va='bottom', fontweight='bold')
ax.set_ylabel('Q-factor (dB)')
ax.set_title(f'NLC Performance Comparison\n{scene.name}, {power_dbm:+.1f} dBm')
ax.axhline(y=0, color='gray', linewidth=0.5, linestyle='--')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## What's Next?

This demo shows the foundation. The full project includes:

- **Multiple scenarios**: MVB-1 (16QAM), MVB-2 (DP-16QAM), MVB-3 (64QAM+PCS)
- **More models**: CNN-NLC, Transformer-NLC, KAN-NLC (Phase 2)
- **Benchmark protocol**: standardized scoring with Q-factor + FLOPs
- **Contributor framework**: decentralized training with contribution tracking

**Get involved:** https://github.com/AI-Fiber-NLC/AI-Fiber-NLC